<a href="https://colab.research.google.com/github/Ranjitkumarmuduli/tomato-disease-detection/blob/main/02_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 2 – Model Training
Fine-tune pretrained CNN models to classify 10 tomato leaf classes (9 diseases + healthy).
Models compared: EfficientNet-B0, MobileNetV3-Large, ResNet-50, and a baseline CNN.

## 1. Install libraries
- `timm`: a library of pretrained image models (EfficientNet, ResNet, MobileNet …)
- `kagglehub`: re-downloads the dataset (Colab storage resets every session)

In [1]:
!pip -q install timm kagglehub

## 2. Import libraries and check the GPU
Training on a GPU is about 20-50x faster than on a CPU. The output must show **Tesla T4**.

In [2]:
import os, time, random                  # files, timing, randomness
import numpy as np, pandas as pd         # numbers and tables
import matplotlib.pyplot as plt          # charts
import torch, torch.nn as nn             # PyTorch deep learning
import timm, kagglehub                   # pretrained models, dataset download
from PIL import Image                    # open image files
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T  # image preprocessing and augmentation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU – switch the runtime to T4 GPU!")

Device: Tesla T4


## 3. Connect Google Drive and load the dataset
The split files from Step 1 are read from Drive; the images are downloaded again from Kaggle.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = "/content/drive/MyDrive/tomato_project"   # same project folder as Step 1
base = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
ROOT = f"{base}/plantvillage dataset/color"       # image folder

CLASSES = pd.read_csv(f"{PROJ}/splits/classes.csv", header=None)[0].tolist()
NUM_CLASSES = len(CLASSES)
print(NUM_CLASSES, "classes:", CLASSES)

Mounted at /content/drive
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
10 classes: ['Bacterial_spot', 'Early_blight', 'Late_blight', 'Leaf_Mold', 'Septoria_leaf_spot', 'Spider_mites Two-spotted_spider_mite', 'Target_Spot', 'Tomato_Yellow_Leaf_Curl_Virus', 'Tomato_mosaic_virus', 'healthy']


## 4. Training settings
**Change `MODEL` here to train another model**, then run Cells 8–12 again. Run them in this order:
1. `efficientnet_b0`
2. `mobilenetv3_large_100`
3. `resnet50`
4. `baseline_cnn`

Training happens in two stages:
- **Stage 1 (head):** only the new final layer learns; the pretrained part is frozen.
- **Stage 2 (full):** the whole network is fine-tuned with a smaller learning rate.

In [4]:
MODEL       = "efficientnet_b0"   # <- change this for each run
IMG_SIZE    = 224                 # images resized to 224x224 pixels (standard for these models)
BATCH_SIZE  = 64                  # images processed together in one step
EPOCHS_HEAD = 2                   # stage 1 epochs (one epoch = one pass over all training images)
EPOCHS_FULL = 8                   # stage 2 epochs
LR_HEAD     = 1e-3                # learning rate, stage 1
LR_FULL     = 3e-4                # smaller learning rate, stage 2 (gentle fine-tuning)
PATIENCE    = 3                   # stop early if validation loss doesn't improve for 3 epochs
SEED        = 42                  # fixed seed = reproducible results

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Training model:", MODEL)

Training model: efficientnet_b0


## 5. Preprocessing and data augmentation
**Augmentation** creates random variations of the training images (crop, flip, rotate, colour change),
so the model learns disease patterns instead of memorising exact photos. This reduces overfitting.
Validation and test images are **not** augmented, only resized.
Normalisation uses ImageNet mean and std, because the pretrained models were trained on ImageNet.